## Assignment 1:
- Implementing Decision Tree using NumPy.
- Train and evaluate this method on the [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset using F1 score.

* Có nhiều thuật toán Decision Tree: ID3, CART, C4.5 -> Lựa chọn CART vì giải quyết cả lẫn bài toán phân loại lẫn hồi quy.

In [14]:
import numpy as np

In [15]:
import numpy as np
import pandas as pd

class CART:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
        self.classes = None

    def _gini(self, y_groups, classes):
        n_instances = float(sum([len(group) for group in y_groups]))
        gini = 0.0
        for group in y_groups:
            size = len(group)
            if size == 0:
                continue
            score = 0.0
            for class_val in classes:
                p = (group == class_val).sum() / size
                score += p * p
            gini += (1.0 - score) * (size / n_instances)
        return gini

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        if not isinstance(y, pd.Series):
            y = pd.Series(y)
       
        X = X.reset_index(drop=True)
        y = y.reset_index(drop=True)
        
        self.classes = np.unique(y)
        
        data = pd.concat([X, y.rename('target')], axis=1)
        
        self.tree = self._build_tree(data, 0)

    def _build_tree(self, data, depth):
        n_samples = len(data)
        n_features = data.shape[1] - 1
        y = data['target']
        classes = np.unique(y)

        if (depth >= self.max_depth) or (n_samples < self.min_samples_split) or (len(classes) <= 1):
            return y.value_counts().idxmax()

        best_gini = float('inf')
        best_split = None

        for feature in data.columns[:-1]:  
            current_col = data[feature]
            thresholds = np.unique(current_col)
            
            for threshold in thresholds:
                left_mask = current_col <= threshold
                left_data = data[left_mask]
                right_data = data[~left_mask]

                if len(left_data) == 0 or len(right_data) == 0:
                    continue

                y_left = left_data['target']
                y_right = right_data['target']
                current_gini = self._gini([y_left, y_right], classes)

                if current_gini < best_gini:
                    best_gini = current_gini
                    best_split = {
                        'feature': feature,
                        'threshold': threshold,
                        'left': left_data,
                        'right': right_data
                    }

        if best_split is None:
            return y.value_counts().idxmax()

        return {
            'feature': best_split['feature'],
            'threshold': best_split['threshold'],
            'left': self._build_tree(best_split['left'], depth + 1),
            'right': self._build_tree(best_split['right'], depth + 1)
        }

    def predict_sample(self, node, sample):
        if not isinstance(node, dict):
            return node
        
        if sample[node['feature']] <= node['threshold']:
            return self.predict_sample(node['left'], sample)
        else:
            return self.predict_sample(node['right'], sample)

    def predict(self, X):
        if self.tree is None:
            raise Exception("Model chưa được huấn luyện!")
        
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
            
        return X.apply(lambda row: self.predict_sample(self.tree, row), axis=1)

In [16]:
data_red = pd.read_csv(r'.\wine+quality\winequality-red.csv', sep = ';')
data_red

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [17]:
data_white = pd.read_csv(r'.\wine+quality\winequality-white.csv', sep = ';')
data_white

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [18]:
from sklearn.model_selection import train_test_split
y_red = data_red['quality']
X_red = data_red.drop('quality', axis = 1)

y_white = data_white['quality']
X_white = data_white.drop('quality', axis = 1)

X_red_train, X_red_test, y_red_train, y_red_test = train_test_split(X_red, y_red, test_size = 0.2, random_state = 42)
X_white_train, X_white_test, y_white_train, y_white_test  = train_test_split(X_white, y_white, test_size = 0.2, random_state = 42)

In [19]:
cart_model_red = CART(max_depth = 5, min_samples_split=2)
cart_model_white = CART(max_depth = 5, min_samples_split=2)
cart_model_red.fit(X_red_train, y_red_train)
cart_model_white.fit(X_white_train, y_white_train)

In [20]:
red_pre = cart_model_red.predict(X_red_test)
white_pre = cart_model_white.predict(X_white_test)

In [21]:
from sklearn.metrics import f1_score
f1_score_red = f1_score(y_red_test, red_pre, average = 'macro')
f1_score_white = f1_score(y_white_test, white_pre, average = 'macro')

print(f1_score_red, f1_score_white)

0.26487894821228153 0.2818110873935484


## Assignment 2:
- Implementing Random Forest using NumPy.
- Train and evaluate this method on the [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset using F1 score.

In [22]:
# Random Forest = Decision Tree + Bagging
# Random Forest có thể lựa chọn giữa Bootstrapping và Feature Randomness
# Boostrapping: Các cây học dựa vào một phần dữ liệu 
# Feature Randomness: Các cây học dựa vào một nhóm feature.
# Ở bài tập này, em lựa chọn các cây học theo phương pháp Feature Randomness với số lượng nhóm nhỏ feature là căn n 
# Có hai cơ chế kết hợp kết quả là: Voting và Averaging. Ở bài tập này em sử dụng Voting

# Ý tưởng xây dựng Random Forest sử dụng các cây con là cây được xây dựng theo thuật toán CART ở trên.


In [23]:
class RandomForest:
    def __init__(self, n_trees=50, max_depth=5, min_samples_split=2, random_state=None):
        """
        Random Forest sử dụng Feature Randomness và Voting
        
        Parameters:
        - n_trees: số lượng cây con (CART)
        - max_depth: độ sâu tối đa của mỗi cây
        - min_samples_split: số sample tối thiểu để split
        - random_state: seed cho reproducibility
        """
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []
        self.selected_features = []  
        
    def fit(self, X, y):
        """Huấn luyện Random Forest"""
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        if not isinstance(y, pd.Series):
            y = pd.Series(y)
            
        np.random.seed(self.random_state)
        
        n_features = X.shape[1]
        n_feature_per_tree = max(1, int(np.sqrt(n_features)))  
        
        for i in range(self.n_trees):
            selected_idx = np.random.choice(n_features, size=n_feature_per_tree, replace=False)
            selected_features = X.columns[selected_idx].tolist()
            
            X_subset = X[selected_features]
            cart = CART(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            cart.fit(X_subset, y)
            
            self.trees.append(cart)
            self.selected_features.append(selected_features)
    
    def predict(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        if len(self.trees) == 0:
            raise Exception("Model chưa được huấn luyện!")
        
        predictions = []
        
        for i, tree in enumerate(self.trees):
            X_subset = X[self.selected_features[i]]
            pred = tree.predict(X_subset).values 
            predictions.append(pred)
        
        predictions = np.array(predictions)
        
        predictions_T = predictions.T 
        
        final_predictions = []
        for sample_votes in predictions_T:
            unique, counts = np.unique(sample_votes, return_counts=True)
            most_common_class = unique[np.argmax(counts)]
            final_predictions.append(most_common_class)
        
        return np.array(final_predictions)

In [24]:

rf_model_red = RandomForest(n_trees=50, max_depth=5, min_samples_split=2, random_state=42)
rf_model_white = RandomForest(n_trees=50, max_depth=5, min_samples_split=2, random_state=42)

rf_model_red.fit(X_red_train, y_red_train)
rf_model_white.fit(X_white_train, y_white_train)

red_pred = rf_model_red.predict(X_red_test)
white_pred = rf_model_white.predict(X_white_test)

f1_red = f1_score(y_red_test, red_pred, average='macro')
f1_white = f1_score(y_white_test, white_pred, average='macro')

print(f1_red)
print(f1_white )

0.24960235019552865
0.2003642746673869


## Assignment 3:
- Train and evaluate the Decision Tree method using a machine learning library.
- Train and evaluate the Random Forest method using a machine learning library.

In [25]:
from sklearn.tree import DecisionTreeClassifier
red_dc = DecisionTreeClassifier(criterion = 'gini', max_depth = 5, min_samples_split = 2, random_state = 42)
white_dc = DecisionTreeClassifier(criterion = 'gini', max_depth = 5, min_samples_split=2, random_state=42)

red_dc.fit(X_red_train, y_red_train)
white_dc.fit(X_white_train, y_white_train)

red_pre = red_dc.predict(X_red_test)
white_pre = white_dc.predict(X_white_test)

f1_score_red = f1_score(y_red_test, red_pre,average = 'macro')
f1_score_white = f1_score(y_white_test, white_pre, average = 'macro')

print(f1_score_red)
print(f1_score_white)

0.264099158919876
0.2827636241066668


In [26]:
from sklearn.ensemble import RandomForestClassifier

red_rf = RandomForestClassifier(n_estimators= 50, criterion = 'gini', random_state = 42)
white_rf = RandomForestClassifier(n_estimators= 50, criterion = 'gini', random_state= 42)

red_rf.fit(X_red_train, y_red_train)
white_rf.fit(X_white_train, y_white_train)

red_pre = red_rf.predict(X_red_test)
white_pre = white_rf.predict(X_white_test)

f1_score_red = f1_score(y_red_test, red_pre, average = 'macro')
f1_score_white = f1_score(y_white_test, white_pre, average = 'macro')

print(f1_score_red)
print(f1_score_white)


0.31352186646391256
0.48949001060029146
